1. Title and objective

# CICIoMT2024 - Preprocessing and Train/Test Split

## Objective
The purpose of this notebook is to prepare the CICIoMT2024 dataset for machine learning. This includes target preparation, feature selection, removal of low-information columns, optional reduction of highly correlated features, creation of model-ready datasets, and stratified train/test splitting for binary and multiclass tasks.

Two preprocessing views are created:

1. A tree-model version, where scaling is not required
2. A linear-model version, where scaling is applied

This notebook uses the merged dataset created earlier and exports processed files for later modeling notebooks.

2. Imports

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

3. Display settings

In [2]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 1000)

4. Load merged dataset

## Load merged dataset

In [3]:
DATA_PATH = Path("../data/interim/ciciomt2024_merged.csv")
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (3385313, 91)


,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Total Fwd Packet,Total Bwd packets,Total Length of Fwd Packet,Total Length of Bwd Packet,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Packet Length Min,Packet Length Max,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWR Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Fwd Segment Size Avg,Bwd Segment Size Avg,Fwd Bytes/Bulk Avg,Fwd Packet/Bulk Avg,Fwd Bulk Rate Avg,Bwd Bytes/Bulk Avg,Bwd Packet/Bulk Avg,Bwd Bulk Rate Avg,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,FWD Init Win Bytes,Bwd Init Win Bytes,Fwd Act Data Pkts,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Attack Name,Label,source_file,attack_label,attack_category,target_binary,target_multiclass_full,target_multiclass_grouped
0,34.173.20.6-192.168.137.234-443-58644-6,34.173.20.6,443,192.168.137.234,58644,6,14/09/2023 09:24:26 AM,14746364,6,0,63.0,0.0,39.0,0.0,10.500000,16.944025,0.0,0.0,0.00000,0.000000,4.272240,0.406880,2.949273e+06,6.558360e+06,14681116.0,2.0,14746364.0,2.949273e+06,6.558360e+06,14681116.0,2.0,0.0,0.000000,0.000000,0.0,0.0,0,0,0,0,192,0,0.406880,0.000000,0.0,39.0,9.000000,15.968719,255.000000,1,0,1,2,6,0,0,0,0.0,10.500000,10.500000,0.00000,0,0,0,0,0,0,6,63,0,0,331,0,2,32,0.0,0.0,0.0,0.0,14681116.0,0.000000,14681116.0,14681116.0,Benign Traffic,0,Benign Traffic.csv,Benign Traffic,Benign,0,Benign Traffic,Benign
1,34.173.20.6-192.168.137.234-443-58646-6,34.173.20.6,443,192.168.137.234,58646,6,14/09/2023 09:24:41 AM,60102110,18,0,4135.0,0.0,1408.0,0.0,229.722222,490.460505,0.0,0.0,0.00000,0.000000,68.799581,0.299490,3.535418e+06,6.531527e+06,15061923.0,1.0,60102110.0,3.535418e+06,6.531527e+06,15061923.0,1.0,0.0,0.000000,0.000000,0.0,0.0,0,0,0,0,584,0,0.299490,0.000000,0.0,1408.0,217.631579,479.546685,229965.023392,1,1,1,7,18,0,0,0,0.0,229.722222,229.722222,0.00000,0,0,0,4072,6,56313,4,1033,0,0,43648,0,8,32,219123.0,0.0,219123.0,219123.0,14957706.0,190543.248665,15061923.0,14672051.0,Benign Traffic,0,Benign Traffic.csv,Benign Traffic,Benign,0,Benign Traffic,Benign
2,52.40.210.103-192.168.137.40-18665-60378-6,52.40.210.103,18665,192.168.137.40,60378,6,14/09/2023 09:24:14 AM,90106769,9,0,1164.0,0.0,194.0,0.0,129.333333,97.000000,0.0,0.0,0.00000,0.000000,12.918008,0.099882,1.126335e+07,1.538550e+07,30049066.0,4.0,90106769.0,1.126335e+07,1.538550e+07,30049066.0,4.0,0.0,0.000000,0.000000,0.0,0.0,1,0,0,0,312,0,0.099882,0.000000,0.0,194.0,135.800000,93.710903,8781.733333,0,0,0,6,9,0,0,0,0.0,150.888889,129.333333,0.00000,0,0,0,0,0,0,3,388,0,0,850,0,5,32,582529.0,0.0,582529.0,582529.0,29841412.0,319090.538141,30049066.0,29473997.0,Benign Traffic,0,Benign Traffic.csv,Benign Traffic,Benign,0,Benign Traffic,Benign
3,10.0.0.3-10.0.0.254-44505-1883-6,10.0.0.3,44505,10.0.0.254,1883,6,14/09/2023 09:24:15 AM,119998980,123,123,2157.0,4.0,20.0,0.0,17.536585,2.309257,2.0,0.0,0.03252,0.253983,18.008486,2.050017,4.897918e+05,5.001567e+05,1008475.0,29.0,119998868.0,9.835973e+05,1.271666e+05,1008531.0,2539.0,119998918.0,983597.688525,127738.757478,1042745.0,104.0,1,0,0,0,3936,3936,1.025009,1.025009,0.0,20.0,8.821862,8.922922,79.618544,0,0,0,123,246,0,0,0,1.0,8.857724,17.536585,0.03252,0,0,0,72,4,24,2,38,2,0,502,64,120,32,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,Benign Traffic,0,Benign Traffic.csv,Benign Traffic,Benign,0,Benign Traffic,Benign
4,10.0.0.7-10.0.0.254-

5. Confirm target columns

In [4]:
print("Binary target distribution:")
print(df["target_binary"].value_counts())

print("\nGrouped multiclass distribution:")
print(df["target_multiclass_grouped"].value_counts())

print("\nFull multiclass distribution:")
print(df["target_multiclass_full"].value_counts().head(20))

Binary target distribution:
target_binary
1    3352693
0      32620
Name: count, dtype: int64

Grouped multiclass distribution:
target_multiclass_grouped
DoS            2112138
MQTT Attack     655143
Recon           579231
Benign           32620
DDoS              5128
Spoofing          1053
Name: count, dtype: int64

Full multiclass distribution:
target_multiclass_full
DoS TCP Flood               2106916
Recon Port Scan              485522
MQTT DDoS Publish Flood      413913
MQTT DoS Connect Flood       238031
Recon OS Scan                 85317
Benign Traffic                32620
Recon Vulnerability Scan       8321
DoS UDP Flood                  3115
DDoS UDP Flood                 2576
DDoS ICMP Flood                2552
MQTT Malformed                 2246
DoS ICMP Flood                 2107
MITM ARP Spoofing              1053
MQTT DoS Publish Flood          953
Recon Ping Sweep                 71
Name: count, dtype: int64


6. Define metadata and target columns

## Define metadata and target columns
These columns are not used as predictors.

In [5]:
metadata_and_target_cols = [
    "Label",
    "Attack Name",
    "attack_label",
    "attack_category",
    "target_binary",
    "target_multiclass_full",
    "target_multiclass_grouped",
    "source_file"
]

existing_metadata_and_target_cols = [col for col in metadata_and_target_cols if col in df.columns]
existing_metadata_and_target_cols

['Label',
 'Attack Name',
 'attack_label',
 'attack_category',
 'target_binary',
 'target_multiclass_full',
 'target_multiclass_grouped',
 'source_file']

7. Create initial feature matrix

In [6]:
X_full = df.drop(columns=existing_metadata_and_target_cols, errors="ignore").copy()

print("Initial feature matrix shape:", X_full.shape)
X_full.head()

Initial feature matrix shape: (3385313, 83)


,Flow ID,Src IP,Src Port,Dst IP,Dst Port,Protocol,Timestamp,Flow Duration,Total Fwd Packet,Total Bwd packets,Total Length of Fwd Packet,Total Length of Bwd Packet,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Packet Length Min,Packet Length Max,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWR Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Fwd Segment Size Avg,Bwd Segment Size Avg,Fwd Bytes/Bulk Avg,Fwd Packet/Bulk Avg,Fwd Bulk Rate Avg,Bwd Bytes/Bulk Avg,Bwd Packet/Bulk Avg,Bwd Bulk Rate Avg,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,FWD Init Win Bytes,Bwd Init Win Bytes,Fwd Act Data Pkts,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min
0,34.173.20.6-192.168.137.234-443-58644-6,34.173.20.6,443,192.168.137.234,58644,6,14/09/2023 09:24:26 AM,14746364,6,0,63.0,0.0,39.0,0.0,10.500000,16.944025,0.0,0.0,0.00000,0.000000,4.272240,0.406880,2.949273e+06,6.558360e+06,14681116.0,2.0,14746364.0,2.949273e+06,6.558360e+06,14681116.0,2.0,0.0,0.000000,0.000000,0.0,0.0,0,0,0,0,192,0,0.406880,0.000000,0.0,39.0,9.000000,15.968719,255.000000,1,0,1,2,6,0,0,0,0.0,10.500000,10.500000,0.00000,0,0,0,0,0,0,6,63,0,0,331,0,2,32,0.0,0.0,0.0,0.0,14681116.0,0.000000,14681116.0,14681116.0
1,34.173.20.6-192.168.137.234-443-58646-6,34.173.20.6,443,192.168.137.234,58646,6,14/09/2023 09:24:41 AM,60102110,18,0,4135.0,0.0,1408.0,0.0,229.722222,490.460505,0.0,0.0,0.00000,0.000000,68.799581,0.299490,3.535418e+06,6.531527e+06,15061923.0,1.0,60102110.0,3.535418e+06,6.531527e+06,15061923.0,1.0,0.0,0.000000,0.000000,0.0,0.0,0,0,0,0,584,0,0.299490,0.000000,0.0,1408.0,217.631579,479.546685,229965.023392,1,1,1,7,18,0,0,0,0.0,229.722222,229.722222,0.00000,0,0,0,4072,6,56313,4,1033,0,0,43648,0,8,32,219123.0,0.0,219123.0,219123.0,14957706.0,190543.248665,15061923.0,14672051.0
2,52.40.210.103-192.168.137.40-18665-60378-6,52.40.210.103,18665,192.168.137.40,60378,6,14/09/2023 09:24:14 AM,90106769,9,0,1164.0,0.0,194.0,0.0,129.333333,97.000000,0.0,0.0,0.00000,0.000000,12.918008,0.099882,1.126335e+07,1.538550e+07,30049066.0,4.0,90106769.0,1.126335e+07,1.538550e+07,30049066.0,4.0,0.0,0.000000,0.000000,0.0,0.0,1,0,0,0,312,0,0.099882,0.000000,0.0,194.0,135.800000,93.710903,8781.733333,0,0,0,6,9,0,0,0,0.0,150.888889,129.333333,0.00000,0,0,0,0,0,0,3,388,0,0,850,0,5,32,582529.0,0.0,582529.0,582529.0,29841412.0,319090.538141,30049066.0,29473997.0
3,10.0.0.3-10.0.0.254-44505-1883-6,10.0.0.3,44505,10.0.0.254,1883,6,14/09/2023 09:24:15 AM,119998980,123,123,2157.0,4.0,20.0,0.0,17.536585,2.309257,2.0,0.0,0.03252,0.253983,18.008486,2.050017,4.897918e+05,5.001567e+05,1008475.0,29.0,119998868.0,9.835973e+05,1.271666e+05,1008531.0,2539.0,119998918.0,983597.688525,127738.757478,1042745.0,104.0,1,0,0,0,3936,3936,1.025009,1.025009,0.0,20.0,8.821862,8.922922,79.618544,0,0,0,123,246,0,0,0,1.0,8.857724,17.536585,0.03252,0,0,0,72,4,24,2,38,2,0,502,64,120,32,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
4,10.0.0.7-10.0.0.254-45727-1883-6,10.0.0.7,45727,10.0.0.254,1883,6,14/09/2023 09:24:15 AM,119998539,123,123,5411.0,4.0,46.0,0.0,43.991870,5.699433,2.0,0.0,0.03252,0.253983,45.125549,2.050025,4.897900e+05,5.001719e+05,1007898.0,9.0,119998518.0,9.835944e+05,1.273177e+05,1007992.0,1432.0,119998515.0,983594.385246,127764.337251,1043184.0,78.0,1,0,0,0,3936,3936,1.025012,1.025012,0.0,46.0,22.101215,22.387612,501.205161,0,0,0,123,246,0,0,0,1.0,22.191057,43.991870,0.03252,0,

In [7]:
numeric_cols = X_full.select_dtypes(include=[np.number]).columns.tolist()
non_numeric_cols = X_full.select_dtypes(exclude=[np.number]).columns.tolist()

print("Numeric feature count:", len(numeric_cols))
print("Non-numeric feature count:", len(non_numeric_cols))
print("Non-numeric columns:", non_numeric_cols)

Numeric feature count: 79
Non-numeric feature count: 4
Non-numeric columns: ['Flow ID', 'Src IP', 'Dst IP', 'Timestamp']


8. Drop non-numeric columns

In [8]:
X_full = X_full.select_dtypes(include=[np.number]).copy()

print("Feature matrix after numeric-only selection:", X_full.shape)

Feature matrix after numeric-only selection: (3385313, 79)


9. Remove zero-variance columns

## Remove zero-variance columns
Columns with no variation do not contribute to predictive learning.

In [9]:
variance_series = X_full.var()
zero_variance_cols = variance_series[variance_series == 0].index.tolist()

print("Zero-variance columns:")
print(zero_variance_cols)
print("\nNumber of zero-variance columns:", len(zero_variance_cols))

Zero-variance columns:
['Protocol', 'Bwd PSH Flags', 'Bwd URG Flags', 'Fwd Bytes/Bulk Avg', 'Fwd Packet/Bulk Avg', 'Fwd Bulk Rate Avg']

Number of zero-variance columns: 6


In [10]:
X_no_zero_var = X_full.drop(columns=zero_variance_cols, errors="ignore").copy()

print("Shape after removing zero-variance columns:", X_no_zero_var.shape)

Shape after removing zero-variance columns: (3385313, 73)


10. Identify near-zero variance columns

## Near-zero variance inspection
These columns are not removed automatically yet, but they are flagged for consideration.

In [11]:
variance_after_zero = X_no_zero_var.var().sort_values()
variance_after_zero.head(20)

Fwd URG Flags               0.000015
URG Flag Count              0.000110
ECE Flag Count              0.000178
CWR Flag Count              0.000201
Fwd PSH Flags               0.013589
FIN Flag Count              0.053686
RST Flag Count              0.440488
SYN Flag Count              0.448994
Down/Up Ratio               2.466842
Fwd Seg Size Min           10.127257
Packet Length Min         643.320683
Subflow Bwd Packets      1232.582925
Subflow Fwd Packets      1727.369942
Fwd Packet Length Std    2108.479685
Bwd Packet Length Std    2874.623136
Bwd Packet Length Min    3423.804682
Bwd Packet/Bulk Avg      3661.339022
Fwd Packet Length Min    3973.505631
Total Bwd packets        4469.098944
PSH Flag Count           4472.848463
dtype: float64

In [12]:
near_zero_variance_cols = variance_after_zero[variance_after_zero < 1e-3].index.tolist()

print("Near-zero variance columns (< 1e-3):")
print(near_zero_variance_cols)
print("\nCount:", len(near_zero_variance_cols))

Near-zero variance columns (< 1e-3):
['Fwd URG Flags', 'URG Flag Count', 'ECE Flag Count', 'CWR Flag Count']

Count: 4


11. Create an optional reduced feature set without near-zero variance columns

In [13]:
X_no_low_var = X_no_zero_var.drop(columns=near_zero_variance_cols, errors="ignore").copy()

print("Shape after removing near-zero variance columns:", X_no_low_var.shape)

Shape after removing near-zero variance columns: (3385313, 69)


12. Identify highly correlated features

## Identify highly correlated features
Highly correlated predictors may introduce redundancy. This matters especially for linear models and interpretability.

In [16]:
corr_matrix = X_no_low_var.corr().abs()

upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr_threshold = 0.95
high_corr_cols = sorted(set(
    col for col in upper_triangle.columns if any(upper_triangle[col] > high_corr_threshold)
))

print("Highly correlated columns to consider dropping (threshold > 0.95):")
print(high_corr_cols)
print("\nCount:", len(high_corr_cols))

Highly correlated columns to consider dropping (threshold > 0.95):
['ACK Flag Count', 'Active Min', 'Average Packet Size', 'Bwd Header Length', 'Bwd Segment Size Avg', 'Flow IAT Max', 'Fwd Header Length', 'Fwd IAT Max', 'Fwd IAT Std', 'Fwd IAT Total', 'Fwd Packets/s', 'Fwd Segment Size Avg', 'Idle Max', 'Packet Length Std', 'Packet Length Variance', 'Subflow Bwd Packets']

Count: 16


At a correlation threshold of 0.95, 16 predictors were identified as highly redundant and excluded from the reduced linear-model feature set. These columns largely reflected near-duplicate representations of packet size, packet rate, inter-arrival time, header length, and activity timing. The full tree-based feature set was retained separately, since tree models can often tolerate correlated inputs more effectively.

In [17]:
high_corr_pairs = []

for col in upper_triangle.columns:
    high_partners = upper_triangle.index[upper_triangle[col] > high_corr_threshold].tolist()
    for partner in high_partners:
        high_corr_pairs.append((partner, col, upper_triangle.loc[partner, col]))

high_corr_pairs_df = pd.DataFrame(high_corr_pairs, columns=["feature_1", "feature_2", "correlation"])
high_corr_pairs_df.sort_values("correlation", ascending=False).head(30)

,feature_1,feature_2,correlation
15,Fwd Packet Length Mean,Fwd Segment Size Avg,1.000000
16,Bwd Packet Length Mean,Bwd Segment Size Avg,1.000000
6,Total Bwd packets,Bwd Header Length,0.999931
1,Flow Duration,Fwd IAT Total,0.998352
7,Flow Packets/s,Fwd Packets/s,0.998286
5,Total Fwd Packet,Fwd Header Length,0.997758
14,Packet Length Mean,Average Packet Size,0.996488
4,Flow IAT Max,Fwd IAT Max,0.994827
20,Flow IAT Max,Idle Max,0.992351
17,Subflow Fwd Packets,Subflow Bwd Packets,0.990049


13. Create feature sets for different model families

## Create model-specific feature sets

Three feature versions are created:

1. `X_tree`: numeric features with zero-variance columns removed
2. `X_linear`: numeric features with zero-variance, near-zero-variance, and highly correlated columns removed
3. `X_linear_scaled`: scaled version of `X_linear`

In [18]:
X_tree = X_no_zero_var.copy()
X_linear = X_no_low_var.drop(columns=high_corr_cols, errors="ignore").copy()

print("X_tree shape:", X_tree.shape)
print("X_linear shape:", X_linear.shape)

X_tree shape: (3385313, 73)
X_linear shape: (3385313, 53)


14. Prepare target vectors

In [19]:
y_binary = df["target_binary"].copy()
y_grouped = df["target_multiclass_grouped"].copy()
y_full = df["target_multiclass_full"].copy()

print("Binary target shape:", y_binary.shape)
print("Grouped multiclass target shape:", y_grouped.shape)
print("Full multiclass target shape:", y_full.shape)

Binary target shape: (3385313,)
Grouped multiclass target shape: (3385313,)
Full multiclass target shape: (3385313,)


15. Encode multiclass targets

## Encode multiclass targets
Tree models can work with integer class labels, so multiclass targets are label-encoded.

In [20]:
grouped_encoder = LabelEncoder()
full_encoder = LabelEncoder()

y_grouped_encoded = pd.Series(
    grouped_encoder.fit_transform(y_grouped),
    name="target_multiclass_grouped_encoded"
)

y_full_encoded = pd.Series(
    full_encoder.fit_transform(y_full),
    name="target_multiclass_full_encoded"
)

print("Grouped class mapping:")
print(dict(zip(grouped_encoder.classes_, grouped_encoder.transform(grouped_encoder.classes_))))

print("\nFull class mapping:")
print(dict(zip(full_encoder.classes_, full_encoder.transform(full_encoder.classes_))))

Grouped class mapping:
{'Benign': np.int64(0), 'DDoS': np.int64(1), 'DoS': np.int64(2), 'MQTT Attack': np.int64(3), 'Recon': np.int64(4), 'Spoofing': np.int64(5)}

Full class mapping:
{'Benign Traffic': np.int64(0), 'DDoS ICMP Flood': np.int64(1), 'DDoS UDP Flood': np.int64(2), 'DoS ICMP Flood': np.int64(3), 'DoS TCP Flood': np.int64(4), 'DoS UDP Flood': np.int64(5), 'MITM ARP Spoofing': np.int64(6), 'MQTT DDoS Publish Flood': np.int64(7), 'MQTT DoS Connect Flood': np.int64(8), 'MQTT DoS Publish Flood': np.int64(9), 'MQTT Malformed': np.int64(10), 'Recon OS Scan': np.int64(11), 'Recon Ping Sweep': np.int64(12), 'Recon Port Scan': np.int64(13), 'Recon Vulnerability Scan': np.int64(14)}


16. Stratified train/test split for binary task

## Binary train/test split
A stratified split preserves the original class imbalance in both training and test sets.

In [21]:
X_train_tree_bin, X_test_tree_bin, y_train_bin, y_test_bin = train_test_split(
    X_tree,
    y_binary,
    test_size=0.2,
    random_state=42,
    stratify=y_binary
)

print("Binary tree-based split shapes:")
print("X_train:", X_train_tree_bin.shape)
print("X_test:", X_test_tree_bin.shape)
print("y_train:", y_train_bin.shape)
print("y_test:", y_test_bin.shape)

Binary tree-based split shapes:
X_train: (2708250, 73)
X_test: (677063, 73)
y_train: (2708250,)
y_test: (677063,)


In [22]:
print("Binary training distribution:")
print(y_train_bin.value_counts(normalize=True))

print("\nBinary test distribution:")
print(y_test_bin.value_counts(normalize=True))

Binary training distribution:
target_binary
1    0.990364
0    0.009636
Name: proportion, dtype: float64

Binary test distribution:
target_binary
1    0.990364
0    0.009636
Name: proportion, dtype: float64


17. Stratified train/test split for grouped multiclass task

In [32]:
X_train_tree_grouped, X_test_tree_grouped, y_train_grouped, y_test_grouped = train_test_split(
    X_tree,
    y_grouped_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_grouped_encoded
)

print("Grouped multiclass tree-based split shapes:")
print("X_train:", X_train_tree_grouped.shape)
print("X_test:", X_test_tree_grouped.shape)
print("y_train:", y_train_grouped.shape)
print("y_test:", y_test_grouped.shape)

Grouped multiclass tree-based split shapes:
X_train: (2708250, 73)
X_test: (677063, 73)
y_train: (2708250,)
y_test: (677063,)


In [33]:
print("Grouped training distribution:")
print(y_train_grouped.value_counts(normalize=True).sort_index())

print("\nGrouped test distribution:")
print(y_test_grouped.value_counts(normalize=True).sort_index())

Grouped training distribution:
target_multiclass_grouped_encoded
0    0.009636
1    0.001515
2    0.623912
3    0.193525
4    0.171101
5    0.000311
Name: proportion, dtype: float64

Grouped test distribution:
target_multiclass_grouped_encoded
0    0.009636
1    0.001515
2    0.623912
3    0.193526
4    0.171101
5    0.000310
Name: proportion, dtype: float64


18. Stratified train/test split for full multiclass task

## Full multiclass split
This split uses all detailed attack labels. Because the class imbalance is severe, model performance should later be evaluated with macro and weighted F1.

In [23]:
X_train_tree_full, X_test_tree_full, y_train_full, y_test_full = train_test_split(
    X_tree,
    y_full_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_full_encoded
)

print("Full multiclass tree-based split shapes:")
print("X_train:", X_train_tree_full.shape)
print("X_test:", X_test_tree_full.shape)
print("y_train:", y_train_full.shape)
print("y_test:", y_test_full.shape)

Full multiclass tree-based split shapes:
X_train: (2708250, 73)
X_test: (677063, 73)
y_train: (2708250,)
y_test: (677063,)


19. Create linear-model train/test splits

## Linear-model splits
A reduced feature set is used for linear models.

In [24]:
X_train_linear_bin, X_test_linear_bin, y_train_linear_bin, y_test_linear_bin = train_test_split(
    X_linear,
    y_binary,
    test_size=0.2,
    random_state=42,
    stratify=y_binary
)

X_train_linear_grouped, X_test_linear_grouped, y_train_linear_grouped, y_test_linear_grouped = train_test_split(
    X_linear,
    y_grouped_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_grouped_encoded
)

X_train_linear_full, X_test_linear_full, y_train_linear_full, y_test_linear_full = train_test_split(
    X_linear,
    y_full_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_full_encoded
)

print("Linear binary train/test:", X_train_linear_bin.shape, X_test_linear_bin.shape)
print("Linear grouped train/test:", X_train_linear_grouped.shape, X_test_linear_grouped.shape)
print("Linear full train/test:", X_train_linear_full.shape, X_test_linear_full.shape)

Linear binary train/test: (2708250, 53) (677063, 53)
Linear grouped train/test: (2708250, 53) (677063, 53)
Linear full train/test: (2708250, 53) (677063, 53)


20. Scale linear-model features

## Scaling for linear models
Tree-based models do not require scaling, but linear and distance-based models usually benefit from it.

In [25]:
scaler_bin = StandardScaler()
scaler_grouped = StandardScaler()
scaler_full = StandardScaler()

X_train_linear_bin_scaled = pd.DataFrame(
    scaler_bin.fit_transform(X_train_linear_bin),
    columns=X_train_linear_bin.columns,
    index=X_train_linear_bin.index
)

X_test_linear_bin_scaled = pd.DataFrame(
    scaler_bin.transform(X_test_linear_bin),
    columns=X_test_linear_bin.columns,
    index=X_test_linear_bin.index
)

X_train_linear_grouped_scaled = pd.DataFrame(
    scaler_grouped.fit_transform(X_train_linear_grouped),
    columns=X_train_linear_grouped.columns,
    index=X_train_linear_grouped.index
)

X_test_linear_grouped_scaled = pd.DataFrame(
    scaler_grouped.transform(X_test_linear_grouped),
    columns=X_test_linear_grouped.columns,
    index=X_test_linear_grouped.index
)

X_train_linear_full_scaled = pd.DataFrame(
    scaler_full.fit_transform(X_train_linear_full),
    columns=X_train_linear_full.columns,
    index=X_train_linear_full.index
)

X_test_linear_full_scaled = pd.DataFrame(
    scaler_full.transform(X_test_linear_full),
    columns=X_test_linear_full.columns,
    index=X_test_linear_full.index
)

print("Scaled binary train/test:", X_train_linear_bin_scaled.shape, X_test_linear_bin_scaled.shape)
print("Scaled grouped train/test:", X_train_linear_grouped_scaled.shape, X_test_linear_grouped_scaled.shape)
print("Scaled full train/test:", X_train_linear_full_scaled.shape, X_test_linear_full_scaled.shape)

Scaled binary train/test: (2708250, 53) (677063, 53)
Scaled grouped train/test: (2708250, 53) (677063, 53)
Scaled full train/test: (2708250, 53) (677063, 53)


21. Optional filtered full multiclass dataset

## Optional filtered full multiclass dataset
Very rare classes can be filtered for an additional experiment.

In [26]:
class_counts = y_full.value_counts()
min_samples_threshold = 1000
valid_full_classes = class_counts[class_counts >= min_samples_threshold].index.tolist()

print("Classes kept with threshold >=", min_samples_threshold)
print(valid_full_classes)

Classes kept with threshold >= 1000
['DoS TCP Flood', 'Recon Port Scan', 'MQTT DDoS Publish Flood', 'MQTT DoS Connect Flood', 'Recon OS Scan', 'Benign Traffic', 'Recon Vulnerability Scan', 'DoS UDP Flood', 'DDoS UDP Flood', 'DDoS ICMP Flood', 'MQTT Malformed', 'DoS ICMP Flood', 'MITM ARP Spoofing']


In [27]:
filtered_mask = y_full.isin(valid_full_classes)

X_full_filtered = X_tree.loc[filtered_mask].copy()
y_full_filtered = y_full.loc[filtered_mask].copy()

filtered_encoder = LabelEncoder()
y_full_filtered_encoded = pd.Series(
    filtered_encoder.fit_transform(y_full_filtered),
    name="target_multiclass_full_filtered_encoded"
)

print("Filtered feature shape:", X_full_filtered.shape)
print("Filtered target distribution:")
print(y_full_filtered.value_counts())

Filtered feature shape: (3384289, 73)
Filtered target distribution:
target_multiclass_full
DoS TCP Flood               2106916
Recon Port Scan              485522
MQTT DDoS Publish Flood      413913
MQTT DoS Connect Flood       238031
Recon OS Scan                 85317
Benign Traffic                32620
Recon Vulnerability Scan       8321
DoS UDP Flood                  3115
DDoS UDP Flood                 2576
DDoS ICMP Flood                2552
MQTT Malformed                 2246
DoS ICMP Flood                 2107
MITM ARP Spoofing              1053
Name: count, dtype: int64


In [28]:
X_train_tree_full_filtered, X_test_tree_full_filtered, y_train_full_filtered, y_test_full_filtered = train_test_split(
    X_full_filtered,
    y_full_filtered_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_full_filtered_encoded
)

print("Filtered full multiclass split:")
print("X_train:", X_train_tree_full_filtered.shape)
print("X_test:", X_test_tree_full_filtered.shape)

Filtered full multiclass split:
X_train: (2707431, 73)
X_test: (676858, 73)


22. Preprocessing summary table

In [29]:
preprocessing_summary = pd.DataFrame({
    "Step": [
        "Initial numeric features",
        "After zero-variance removal",
        "After near-zero variance removal",
        "After high-correlation removal (linear set)",
        "Binary target created",
        "Grouped multiclass target created",
        "Full multiclass target created",
        "Scaled linear datasets created"
    ],
    "Result": [
        X_full.shape[1],
        X_no_zero_var.shape[1],
        X_no_low_var.shape[1],
        X_linear.shape[1],
        "Yes",
        "Yes",
        "Yes",
        "Yes"
    ]
})

preprocessing_summary

,Step,Result
0,Initial numeric features,79
1,After zero-variance removal,73
2,After near-zero variance removal,69
3,After high-correlation removal (linear set),53
4,Binary target created,Yes
5,Grouped multiclass target created,Yes
6,Full multiclass target created,Yes
7,Scaled linear datasets created,Yes


23. Save processed datasets

## Save processed datasets
Processed datasets are exported for use in the modeling notebooks.

In [30]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [34]:
X_train_tree_bin.to_csv(PROCESSED_DIR / "X_train_tree_binary.csv", index=False)
X_test_tree_bin.to_csv(PROCESSED_DIR / "X_test_tree_binary.csv", index=False)
y_train_bin.to_csv(PROCESSED_DIR / "y_train_binary.csv", index=False)
y_test_bin.to_csv(PROCESSED_DIR / "y_test_binary.csv", index=False)

X_train_tree_grouped.to_csv(PROCESSED_DIR / "X_train_tree_grouped.csv", index=False)
X_test_tree_grouped.to_csv(PROCESSED_DIR / "X_test_tree_grouped.csv", index=False)
y_train_grouped.to_csv(PROCESSED_DIR / "y_train_grouped.csv", index=False)
y_test_grouped.to_csv(PROCESSED_DIR / "y_test_grouped.csv", index=False)

X_train_tree_full.to_csv(PROCESSED_DIR / "X_train_tree_full.csv", index=False)
X_test_tree_full.to_csv(PROCESSED_DIR / "X_test_tree_full.csv", index=False)
y_train_full.to_csv(PROCESSED_DIR / "y_train_full.csv", index=False)
y_test_full.to_csv(PROCESSED_DIR / "y_test_full.csv", index=False)

X_train_linear_bin_scaled.to_csv(PROCESSED_DIR / "X_train_linear_binary_scaled.csv", index=False)
X_test_linear_bin_scaled.to_csv(PROCESSED_DIR / "X_test_linear_binary_scaled.csv", index=False)

X_train_linear_grouped_scaled.to_csv(PROCESSED_DIR / "X_train_linear_grouped_scaled.csv", index=False)
X_test_linear_grouped_scaled.to_csv(PROCESSED_DIR / "X_test_linear_grouped_scaled.csv", index=False)

X_train_linear_full_scaled.to_csv(PROCESSED_DIR / "X_train_linear_full_scaled.csv", index=False)
X_test_linear_full_scaled.to_csv(PROCESSED_DIR / "X_test_linear_full_scaled.csv", index=False)

print("Processed datasets saved successfully.")

Processed datasets saved successfully.


24. Save encoders and feature lists

In [35]:
REPORT_DIR = Path("../reports/results")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

pd.DataFrame({"feature": X_tree.columns}).to_csv(REPORT_DIR / "tree_features.csv", index=False)
pd.DataFrame({"feature": X_linear.columns}).to_csv(REPORT_DIR / "linear_features.csv", index=False)

pd.DataFrame({
    "grouped_class_name": grouped_encoder.classes_,
    "grouped_class_code": grouped_encoder.transform(grouped_encoder.classes_)
}).to_csv(REPORT_DIR / "grouped_label_mapping.csv", index=False)

pd.DataFrame({
    "full_class_name": full_encoder.classes_,
    "full_class_code": full_encoder.transform(full_encoder.classes_)
}).to_csv(REPORT_DIR / "full_label_mapping.csv", index=False)

high_corr_pairs_df.to_csv(REPORT_DIR / "high_correlation_pairs.csv", index=False)
preprocessing_summary.to_csv(REPORT_DIR / "preprocessing_summary.csv", index=False)

print("Feature lists and mappings saved successfully.")

Feature lists and mappings saved successfully.


25. Final checks

In [36]:
print("Final tree feature count:", X_tree.shape[1])
print("Final linear feature count:", X_linear.shape[1])

print("\nBinary train/test shapes:")
print(X_train_tree_bin.shape, X_test_tree_bin.shape)

print("\nGrouped multiclass train/test shapes:")
print(X_train_tree_grouped.shape, X_test_tree_grouped.shape)

print("\nFull multiclass train/test shapes:")
print(X_train_tree_full.shape, X_test_tree_full.shape)

Final tree feature count: 73
Final linear feature count: 53

Binary train/test shapes:
(2708250, 73) (677063, 73)

Grouped multiclass train/test shapes:
(2708250, 73) (677063, 73)

Full multiclass train/test shapes:
(2708250, 73) (677063, 73)


26. Final summary

## Summary of preprocessing

The dataset was converted into model-ready form by removing non-numeric columns and excluding zero-variance features. Near-zero variance features were identified and removed in a reduced feature set, while a correlation-based reduction step was applied for linear-model preparation in order to reduce redundancy and multicollinearity.

Three supervised learning targets were prepared: binary attack detection, grouped multiclass attack classification, and full multiclass attack-type classification. Stratified train/test splits were used throughout to preserve the original class distributions. Separate scaled datasets were also created for linear models.

This notebook now provides the processed inputs required for baseline model training and comparison in the next stage of the project.

### Methodological note
Highly correlated features were not removed from the tree-based feature set because tree ensembles can often handle redundant predictors effectively. However, a reduced feature version was created for linear models to improve stability, interpretability, and computational efficiency.